# Homework: Sentiment Steering and Sparse Autoencoders

This assignment explores two mechanistic interpretability techniques:

1. Sentiment steering via activation addition.
2. Sparse autoencoder on model activations.

In [1]:
!pip install -q transformer_lens==2.16.1 circuitsvis==1.43.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.0/192.0 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 115.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.

In [2]:
!pip install numpy==2.2.6

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 59.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformer-lens 2.16.1 requires numpy<2,>=1.26; python_version == "3.12", but you have numpy 2.2.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.6 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.


In [1]:
import torch
from transformer_lens import HookedTransformer
import numpy as np

model = HookedTransformer.from_pretrained('gpt2')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loaded pretrained model gpt2 into HookedTransformer
Moving model to device:  cuda


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint()
  (blocks): ModuleList(
    (0-11): 12 x TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): Attention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
      )
      (mlp): MLP(
        (hook_pre): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_attn_out): HookPoint()
      (hook_mlp_out): HookPoint()
      (h

## Part 1 – Sentiment Steering via residual (3 points)

Your task is to steer model into good/bad generations and see the results.

In [2]:
from typing import List
# This is enough to steer! You may experiment with dataset as you want
positive_sentences = [
    'I love this product, it works wonderfully!',
    'This is the best day I have ever had.',
    'I am feeling fantastic and everything is great.',
    'What a delightful surprise!',
    'The food was amazing and the service was excellent.'
]

negative_sentences = [
    'I hate this product, it is terrible.',
    'This is the worst day of my life.',
    'I am feeling awful and everything is bad.',
    'What a horrible experience.',
    'The food was disgusting and the service was terrible.'
]

# Function to collect average residual activations for a list of sentences
def collect_average_residuals(sent_list: List[str]):
    # Hint: You need *residual pre hook* from each layer
    avgs = []
    for layer in range(model.cfg.n_layers):
      acts = []

      for text in sent_list:
        _, cache = model.run_with_cache(text)
        resid_pre = cache[f"blocks.{layer}.hook_resid_pre"]
        acts.append(resid_pre.mean(dim=1))
      avgs.append(torch.stack(acts).mean(dim=0))
    return avgs

pos_avgs = collect_average_residuals(positive_sentences)
neg_avgs = collect_average_residuals(negative_sentences)

steering_vectors = [pos - neg for pos, neg in zip(pos_avgs, neg_avgs)]

In [3]:
# Function to generate text with steering applied
def generate_with_steering(prompt, max_new_tokens=20, coef=0.0):
    tokens = model.to_tokens(prompt).to(device)

    for _ in range(max_new_tokens):
        # Define hooks that add the steering vector times coef at every layer
        hooks = []
        # Hint: make hooks that add steering vecor with coef to each layer
        for layer in range(model.cfg.n_layers):
          hooks.append((f"blocks.{layer}.hook_resid_pre", lambda resid_pre, hook, v=steering_vectors[layer]: resid_pre + coef * v))

        logits = model.run_with_hooks(tokens, fwd_hooks=hooks)
        next_token = logits[0, -1].argmax().unsqueeze(0)
        tokens = torch.cat([tokens, next_token.unsqueeze(0)], dim=1)

        if next_token == model.tokenizer.eos_token_id:
            break

    return model.to_string(tokens[0, 1:])

prompt = 'The movie that I watched yesterday was'
print('Neutral completion:')
print(generate_with_steering(prompt, max_new_tokens=20, coef=0.0))
print('Positive-steered completion:')
print(generate_with_steering(prompt, max_new_tokens=20, coef=0.3))
print('Negative-steered completion:')
print(generate_with_steering(prompt, max_new_tokens=20, coef=-0.3))

Neutral completion:
The movie that I watched yesterday was a bit of a disappointment. I was hoping for a more mature, more mature, more mature movie
Positive-steered completion:
The movie that I watched yesterday was a great one. I was able to watch it with my wife and her family. I was able
Negative-steered completion:
The movie that I watched yesterday was a horrible movie. It was a horrible movie. It was a horrible movie. It was a horrible


You should see how this is much more effective than tinkering with attantion heads from seminar.

## Part 2 – Sparse Autoencoder on Residual Activations (4 + bonus)

This is compute intensive part and you may adjust hyperparameters to your liking. Try to get meaningful results but in the end it might be compute bound. You still can get max points.


In [4]:
from datasets import load_dataset
ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
texts = ds["text"]

texts = [t for t in texts if len(t.strip()) > 0]
dataset_sentences = texts[:] # You may want to adjust heres

print("Total lines:", len(dataset_sentences))

README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Total lines: 23767


In [ ]:
from tqdm import tqdm

batch_size = 16

last_layer = model.cfg.n_layers - 1

activations = []
token_to_sentence = []
batch_acts = []
with torch.no_grad():
    for sent_id, sent in tqdm(enumerate(dataset_sentences)):
        tokens = model.to_tokens(sent).to(device)
        _, cache = model.run_with_cache(tokens)
        # Take the *residual stream* at the last layer
        resid = cache[f"blocks.{last_layer}.hook_resid_pre"]
        resid = resid.squeeze(0).cpu()

        batch_acts.append(resid)
        token_to_sentence.extend([sent_id] * resid.shape[0]) # store sent_id per activated token

        if (sent_id + 1) % batch_size == 0:
          activations.append(torch.cat(batch_acts, dim=0))
          batch_acts = []

    activations = torch.cat(activations + batch_acts, dim=0)
    token_to_sentence = torch.tensor(token_to_sentence)
    print('Activation dataset shape:', activations.shape)

23767it [15:36, 25.37it/s]


#### Your task is to use any SAE try to disentangle features from residual. In the end you will look at top sentences that activate certain features.

Classic approach is enc + relu + dec \
For loss: mse + coef * l1 on hiddden \
But feel free to experiment!

In [ ]:
import torch.nn as nn

class SAE(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        self.enc = nn.Linear(input_dim, hidden_dim)
        self.dec = nn.Linear(hidden_dim, input_dim)

    def forward(self, x):
        h = torch.relu(self.enc(x))
        out = self.dec(h)
        return out, h

In [ ]:
import torch.optim as optim

input_dim = activations.shape[1]
hidden_dim = input_dim * 2
learning_rate = 1e-3
num_epochs = 500

sae = SAE(input_dim, hidden_dim).to(device)
optimizer = optim.Adam(sae.parameters(), lr=learning_rate)
mse = nn.MSELoss()

l1 = 1e-3
x = activations.to(device)
for epoch in range(num_epochs):
    optimizer.zero_grad()

    recon, h = sae(x)
    loss = mse(recon, x) + l1 * h.abs().mean()

    loss.backward()
    optimizer.step()
    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.6f}')

Below is some code that prints tokens that activate certain neuron the most. You are free to change it!

In [ ]:
sae.eval()
with torch.no_grad():
    _, h = sae(activations_device)
    h = h.cpu().numpy()

# For selected neuron indices, find the sentences with highest activation
selected_neurons = [0, 1, 2]  # you can choose other indices


for neuron in selected_neurons:
    activations_neuron = h[:, neuron]
    top_idx = activations_neuron.argsort()[::-1][:5]
    print(f'Neuron {neuron}:')
    for rank, idx in enumerate(top_idx, start=1):
        sent_id = token_to_sentence[idx].item()
        text = dataset_sentences[sent_id]

        toks = model.to_tokens(text)[0, 1:]
        str_toks = model.to_str_tokens(toks)

        token_positions = (token_to_sentence == sent_id).nonzero(as_tuple=True)[0]
        local_pos = (token_positions == idx).nonzero(as_tuple=True)[0].item()
        cropped = "".join(str_toks[:local_pos + 1])

        print(f"Top {rank}: global token {idx}, sentence {sent_id}, local token {local_pos}")
        print(cropped)
        print()